In [1]:
# ============================================
# CELL 1 — УСТАНОВКА ЗАВИСИМОСТЕЙ
# ============================================

!pip -q install fastapi uvicorn python-multipart
!pip -q install faiss-cpu sentence-transformers
!pip -q install pymupdf python-docx openai nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.1 MB/s eta 0:00:00


In [4]:
# ============================================
# CELL 2 — КОНФИГУРАЦИЯ
# ============================================

import os
import numpy as np
import faiss

from google.colab import userdata
from sentence_transformers import SentenceTransformer
from openai import OpenAI


# --------------------------------------------
# PROXYAPI
# --------------------------------------------

# Получаем секретный ключ из Google Colab Secrets
PROXY_API_KEY = userdata.get("PROXY_API_KEY")

if not PROXY_API_KEY:
    raise RuntimeError(
        "Секрет PROXY_API_KEY не найден. "
        "Добавьте его в Google Colab → Secrets."
    )

PROXY_API_BASE = "https://api.proxyapi.ru/openai/v1"

# Используемая LLM
LLM_MODEL = "gpt-4o-mini"


# --------------------------------------------
# ОГРАНИЧЕНИЯ
# --------------------------------------------

# Максимальный размер файла — 1 МБ
MAX_FILE_SIZE = 1 * 1024 * 1024

# Сколько релевантных фрагментов передавать LLM
TOP_K = 5


# --------------------------------------------
# EMBEDDINGS
# --------------------------------------------

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Загрузка embedding-модели...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)


# --------------------------------------------
# OPENAI-COMPATIBLE CLIENT
# --------------------------------------------

llm_client = OpenAI(
    api_key=PROXY_API_KEY,
    base_url=PROXY_API_BASE
)


# --------------------------------------------
# ГЛОБАЛЬНОЕ СОСТОЯНИЕ RAG
# --------------------------------------------

document_name = None

chunks = []

metadata = []

vector_index = None


print()
print("Backend configuration ready.")
print(f"LLM: {LLM_MODEL}")
print("Embeddings: local")
print("Vector DB: FAISS")
print("Max file size: 1 MB")
print("ProxyAPI key: loaded from Colab Secrets")

Загрузка embedding-модели...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Backend configuration ready.
LLM: gpt-4o-mini
Embeddings: local
Vector DB: FAISS
Max file size: 1 MB
ProxyAPI key: loaded from Colab Secrets


In [5]:
# ============================================
# CELL 3 — RAG + FASTAPI
# ============================================

import io
import re
import fitz

from typing import List
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from docx import Document

# --------------------------------------------
# FASTAPI
# --------------------------------------------

app = FastAPI(
    title="RAG Knowledge Base API",
    version="1.0.0"
)

# Для тестирования frontend из Google AI Studio
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


# --------------------------------------------
# REQUEST MODEL
# --------------------------------------------

class QueryRequest(BaseModel):
    question: str


# --------------------------------------------
# TEXT EXTRACTION
# --------------------------------------------

def extract_pdf(data: bytes):
    pages = []

    pdf = fitz.open(stream=data, filetype="pdf")

    for page_number, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()

        if text:
            pages.append({
                "page": page_number,
                "text": text
            })

    pdf.close()
    return pages


def extract_txt(data: bytes):
    text = data.decode("utf-8", errors="ignore").strip()

    if not text:
        return []

    return [{
        "page": 1,
        "text": text
    }]


def extract_docx(data: bytes):
    document = Document(io.BytesIO(data))

    paragraphs = []

    for paragraph in document.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    text = "\n".join(paragraphs).strip()

    if not text:
        return []

    return [{
        "page": 1,
        "text": text
    }]


def extract_text(filename: str, data: bytes):

    extension = filename.lower().split(".")[-1]

    if extension == "pdf":
        return extract_pdf(data)

    if extension == "txt":
        return extract_txt(data)

    if extension == "docx":
        return extract_docx(data)

    raise HTTPException(
        status_code=400,
        detail="Поддерживаются только PDF, TXT и DOCX."
    )


# --------------------------------------------
# CHUNKING
# --------------------------------------------

def split_text(text: str, chunk_size=900, overlap=150):

    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    result = []
    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunk = text[start:end].strip()

        if chunk:
            result.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return result


# --------------------------------------------
# CREATE VECTOR INDEX
# --------------------------------------------

def create_index(document_pages, filename):

    global chunks
    global metadata
    global vector_index
    global document_name

    chunks = []
    metadata = []
    document_name = filename

    for page_data in document_pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        page_chunks = split_text(page_text)

        for chunk in page_chunks:

            chunks.append(chunk)

            metadata.append({
                "document": filename,
                "page": page_number,
                "chunk": len(chunks)
            })

    if not chunks:
        raise HTTPException(
            status_code=400,
            detail="В документе не найден текст."
        )

    # Создаём embeddings локально
    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings = embeddings.astype("float32")

    dimension = embeddings.shape[1]

    # Inner Product + normalized embeddings = cosine similarity
    vector_index = faiss.IndexFlatIP(dimension)

    vector_index.add(embeddings)

    return {
        "document": filename,
        "chunks": len(chunks)
    }


# --------------------------------------------
# RAG SEARCH
# --------------------------------------------

def search_context(question: str):

    if vector_index is None:
        raise HTTPException(
            status_code=400,
            detail="Сначала загрузите документ."
        )

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indexes = vector_index.search(
        question_embedding,
        min(TOP_K, len(chunks))
    )

    context = []
    sources = []

    for score, index in zip(scores[0], indexes[0]):

        if index < 0:
            continue

        context.append(chunks[index])

        source = metadata[index].copy()
        source["similarity"] = round(float(score), 4)

        sources.append(source)

    return context, sources


# --------------------------------------------
# LLM
# --------------------------------------------

SYSTEM_PROMPT = """
Ты — система поиска информации по документу.

КРИТИЧЕСКИЕ ПРАВИЛА:

1. Отвечай ТОЛЬКО на основании предоставленного контекста.
2. Не используй свои знания для дополнения ответа.
3. Не придумывай отсутствующие факты.
4. Не делай предположений.
5. Не интерпретируй информацию так, чтобы изменить её смысл.
6. Если контекст не содержит ответа на вопрос, обязательно ответь:

"Информация не найдена в загруженном документе."

Ответ должен быть кратким и точным.
"""


def generate_answer(question, context):

    context_text = "\n\n--- ФРАГМЕНТ ---\n\n".join(context)

    user_prompt = f"""
КОНТЕКСТ ДОКУМЕНТА:

{context_text}

ВОПРОС:

{question}

Ответь строго по контексту.
"""

    try:

        response = llm_client.chat.completions.create(
            model=LLM_MODEL,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

        return response.choices[0].message.content.strip()

    except Exception as e:

        raise HTTPException(
            status_code=502,
            detail=f"Ошибка ProxyAPI/LLM: {str(e)}"
        )


# --------------------------------------------
# HEALTH
# --------------------------------------------

@app.get("/health")
def health():

    return {
        "status": "ok",
        "document_loaded": vector_index is not None,
        "document": document_name
    }


# --------------------------------------------
# UPLOAD
# --------------------------------------------

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):

    global vector_index

    allowed_extensions = {"pdf", "txt", "docx"}

    filename = file.filename or ""

    extension = filename.lower().split(".")[-1]

    if extension not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail="Поддерживаются только PDF, TXT и DOCX."
        )

    data = await file.read()

    if len(data) > MAX_FILE_SIZE:

        raise HTTPException(
            status_code=413,
            detail="Файл слишком большой. Максимальный размер — 1 МБ."
        )

    if len(data) == 0:

        raise HTTPException(
            status_code=400,
            detail="Файл пустой."
        )

    pages = extract_text(filename, data)

    result = create_index(pages, filename)

    return {
        "status": "success",
        "message": "Документ успешно обработан и проиндексирован.",
        "document": result["document"],
        "chunks": result["chunks"]
    }


# --------------------------------------------
# QUERY
# --------------------------------------------

@app.post("/query")
def query(request: QueryRequest):

    question = request.question.strip()

    if not question:

        raise HTTPException(
            status_code=400,
            detail="Вопрос не может быть пустым."
        )

    if vector_index is None:

        raise HTTPException(
            status_code=400,
            detail="Сначала загрузите документ."
        )

    context, sources = search_context(question)

    answer = generate_answer(
        question,
        context
    )

    return {
        "answer": answer,
        "sources": sources
    }


print("FastAPI application created successfully.")
print("Endpoints:")
print("GET  /health")
print("POST /upload")
print("POST /query")

FastAPI application created successfully.
Endpoints:
GET  /health
POST /upload
POST /query


In [9]:
# ============================================
# CELL 4 — CLOUDFLARE TUNNEL
# ============================================

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /content/cloudflared

!chmod +x /content/cloudflared

import subprocess
import time
import re

# Запускаем tunnel к уже работающему FastAPI
process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8000",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None

for _ in range(30):
    line = process.stdout.readline()

    if line:
        print(line.strip())

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            line
        )

        if match:
            public_url = match.group(0)
            break

    time.sleep(1)

if public_url:
    print("\n" + "=" * 60)
    print("RAG BACKEND PUBLIC URL")
    print("=" * 60)
    print(public_url)
    print()
    print("Swagger:")
    print(public_url + "/docs")
    print()
    print("Health:")
    print(public_url + "/health")
    print("=" * 60)
else:
    print("Не удалось получить Cloudflare Tunnel URL.")

2026-08-15T14:46:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-15T14:46:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-15T14:46:27Z INF +--------------------------------------------------------------------------------------------+
2026-08-15T14:46:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-15T14:46:27Z INF |  https://suggesting-footage-calibration-surprised.tryc